<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/agent/gemma4_content_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 4 Content Agent — Canva Designs & Descript Video Editing

This notebook adds **content creation tools** to a local Gemma 4 agent:

- **Canva MCP** — agent writes copy → Canva generates the visual design (social posts, flyers, presentations, docs, logos, and more)
- **Descript MCP** — agent writes a script or edit instructions → Descript edits your video/audio by text

**Full pipeline demonstrated at the end:**
```
Gemma 4 writes copy
  → Canva builds the design
    → Bitly shortens the share link
      → Twilio texts it to you
```

**Stack:**
- `gemma4:12b` via Ollama (local LLM, writes all copy and scripts)
- Canva MCP (design generation and editing)
- Descript MCP (video/audio editing by text)
- `llama-index-tools-mcp` (MCP bridge)

## Prerequisites

1. **Ollama** with Gemma 4: `ollama serve && ollama pull gemma4:12b`
2. **Canva account** — connected via the Canva MCP server (already configured if using Claude Code)
3. **Descript account** — connected via the Descript MCP server (same)

In [ ]:
!pip install llama-index-llms-ollama llama-index-tools-mcp llama-index-core

In [ ]:
from llama_index.llms.ollama import Ollama
from llama_index.core.tools import FunctionTool
from llama_index.core.agent.workflow import ReActAgent
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
from typing import Annotated

llm = Ollama(model="gemma4:12b", request_timeout=180.0)
print("LLM ready.")

## Part 1 — Canva Design Agent

### 1.1 Connect to the Canva MCP server

In [ ]:
# The Canva MCP server is available as a hosted service.
# Replace the URL below with your Canva MCP endpoint.
# If using Claude Code on the web, it may already be configured as a session tool.
CANVA_MCP_URL = "https://mcp.canva.com/mcp"  # adjust if self-hosted

canva_client = BasicMCPClient(CANVA_MCP_URL)
canva_spec = McpToolSpec(
    client=canva_client,
    allowed_tools=[
        "generate-design",
        "create-design-from-candidate",
        "get-design",
        "export-design",
        "search-designs",
        "get-design-thumbnail",
    ],
)

canva_tools = await canva_spec.to_tool_list_async()
print(f"Loaded {len(canva_tools)} Canva tools:")
for t in canva_tools:
    print(f"  • {t.metadata.name}")

### 1.2 Copy-first design workflow

Gemma 4 writes the copy, then passes it to Canva to build the design. This keeps the creative brief grounded in your own words rather than generic templates.

In [ ]:
# Step 1: Ask Gemma to write the copy
copy_prompt = """
Write a concise Instagram post caption and headline for a crypto portfolio tracker app.
Tone: confident, modern, slightly technical but accessible.
Include: a hook, one key benefit, and a call to action.
Max 150 characters for the headline. Caption max 200 characters.
"""

copy_response = llm.complete(copy_prompt)
print("Generated copy:")
print(copy_response)

In [ ]:
# Step 2: Feed the copy to Canva to generate a design
canva_agent = ReActAgent(
    tools=canva_tools,
    llm=llm,
    system_prompt=(
        "You are a design assistant. When given copy and a design type, "
        "use generate-design to create the design, then create-design-from-candidate "
        "to save the best candidate to the user's Canva account. "
        "Always report the final design ID and any share URL."
    ),
    verbose=True,
)

design_response = await canva_agent.run(
    f"Create an Instagram post design using this copy:\n\n{copy_response}\n\n"
    "Use a bold, modern dark-theme style. Save the best candidate to my account."
)
print(design_response)

### 1.3 Design type examples

Canva supports 25+ design types. Here are the most useful ones for each use case.

In [ ]:
# Social media post
result = await canva_agent.run(
    "Write and design a Twitter/X post announcing a new LlamaIndex + Gemma 4 integration. "
    "Tech community audience. Include relevant hashtags in the copy."
)
print(result)

In [ ]:
# Business document
result = await canva_agent.run(
    "Write and create a one-page business proposal doc for a local AI consulting service "
    "that helps small businesses run local LLMs privately. Professional, clean style."
)
print(result)

In [ ]:
# Flyer
result = await canva_agent.run(
    "Design a flyer for a local AI meetup group. "
    "Event: 'Running AI Locally' workshop. Date: Saturday July 12. Location: TBD. "
    "Write all copy and design the flyer. Bold, tech-themed style."
)
print(result)

In [ ]:
# Email newsletter
result = await canva_agent.run(
    "Write and design a short weekly email newsletter called 'Local AI Weekly'. "
    "This week's topic: Gemma 4 multimodal capabilities. "
    "Include: intro, 3 bullet highlights, and a footer CTA. Professional minimal style."
)
print(result)

### 1.4 RAG-grounded design

Gemma 4 reads your personal documents (from the private RAG index) and creates designs grounded in your actual data.

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = llm

try:
    storage_context = StorageContext.from_defaults(persist_dir="./my_rag_index")
    rag_index = load_index_from_storage(storage_context)
    print("Loaded personal RAG index.")
except Exception:
    from llama_index.core import VectorStoreIndex, Document
    rag_index = VectorStoreIndex.from_documents([Document(text="""
        Personal finance summary June 2026:
        Savings: $12,800 / $15,000 goal (85% complete, $2,200 gap)
        Crypto: 0.8 ETH + 0.02 BTC
        Next review: July 15, 2026
    """)])
    print("Using demo index.")

rag_tool = QueryEngineTool.from_defaults(
    query_engine=rag_index.as_query_engine(),
    name="personal_documents",
    description="Search personal financial documents and notes.",
)

rag_design_agent = ReActAgent(
    tools=[rag_tool] + canva_tools,
    llm=llm,
    system_prompt=(
        "You are a personal finance design assistant. "
        "Query personal_documents to get real data, write copy based on that data, "
        "then use generate-design to create an infographic or report."
    ),
    verbose=True,
)

result = await rag_design_agent.run(
    "Look up my savings progress and create an infographic showing "
    "my goal vs current balance with a motivational message."
)
print(result)

---
## Part 2 — Descript Video Editing Agent

Descript lets you edit video and audio by editing text — trim, rearrange, remove filler words, add captions, and more. Gemma 4 writes the edit instructions; Descript executes them.

### 2.1 Connect to the Descript MCP server

In [ ]:
DESCRIPT_MCP_URL = "https://mcp.descript.com/mcp"  # adjust to your Descript MCP endpoint

descript_client = BasicMCPClient(DESCRIPT_MCP_URL)
descript_spec = McpToolSpec(
    client=descript_client,
    allowed_tools=[
        "list_projects",
        "get_project",
        "import_media",
        "prompt_project_agent",
        "wait_for_job",
        "publish_project",
    ],
)

descript_tools = await descript_spec.to_tool_list_async()
print(f"Loaded {len(descript_tools)} Descript tools:")
for t in descript_tools:
    print(f"  • {t.metadata.name}")

### 2.2 Script-to-video workflow

Gemma 4 writes a tight script; you record it; Descript handles the edit automatically.

In [ ]:
# Step 1: Gemma writes the script
script_prompt = """
Write a 60-second YouTube tutorial script explaining what Gemma 4 is and how to run it locally with Ollama.
Structure:
- Hook (0-5s): one surprising fact
- What is Gemma 4 (5-20s)
- How to install Ollama + pull gemma4:12b (20-45s) — include the exact commands
- First query demo (45-55s)
- CTA to subscribe (55-60s)
Keep it conversational. Mark each section with [SECTION: name] tags.
"""

script = llm.complete(script_prompt)
print("Generated script:")
print(script)

In [ ]:
# Step 2: Create a Descript agent that uses the script to edit a project
descript_agent = ReActAgent(
    tools=descript_tools,
    llm=llm,
    system_prompt=(
        "You are a video editing assistant powered by Descript. "
        "Use list_projects to find the user's project, get_project to inspect it, "
        "then prompt_project_agent with natural language edit instructions. "
        "Use wait_for_job after any long-running operation. "
        "When done, use publish_project to get a shareable URL."
    ),
    verbose=True,
)

# List available projects first
result = await descript_agent.run("List my Descript projects so I can pick one to edit.")
print(result)

In [ ]:
# Replace with your actual project ID from the list above
PROJECT_ID = "YOUR_PROJECT_ID_HERE"

# Apply edits using natural language
result = await descript_agent.run(
    f"In project {PROJECT_ID}, do the following edits:\n"
    "1. Remove all filler words (um, uh, like, you know)\n"
    "2. Add captions\n"
    "3. Trim any silence longer than 1.5 seconds\n"
    "Then publish as a 1080p video and give me the share URL."
)
print(result)

### 2.3 Script-driven edit

Use the script Gemma wrote to guide Descript on rearranging or trimming specific sections.

In [ ]:
result = await descript_agent.run(
    f"In project {PROJECT_ID}, use this script as a guide to restructure the video:\n\n"
    f"{script}\n\n"
    "Trim any sections not covered by the script. "
    "Add section title cards matching the [SECTION: name] markers. "
    "Publish as 1080p and return the share URL."
)
print(result)

### 2.4 Import and edit new media

In [ ]:
# Import a video file from a URL and edit it immediately
MEDIA_URL = "https://example.com/your-recording.mp4"  # replace with your file

result = await descript_agent.run(
    f"Import this video into a new Descript project: {MEDIA_URL}\n"
    "Once imported, remove filler words, add captions, and publish as 720p."
)
print(result)

---
## Part 3 — Full pipeline: Write → Design → Shorten → Text

All tools from PRs #70, #72, #74 chained together in one agent run.

In [ ]:
import os
from twilio.rest import Client as TwilioClient
from typing import Annotated

TWILIO_ACCOUNT_SID = os.environ.get("TWILIO_ACCOUNT_SID", "")
TWILIO_AUTH_TOKEN  = os.environ.get("TWILIO_AUTH_TOKEN", "")
TWILIO_FROM        = os.environ.get("TWILIO_FROM_NUMBER", "")
MY_PHONE           = os.environ.get("MY_PHONE_NUMBER", "")
BITLY_TOKEN        = os.environ.get("BITLY_TOKEN", "")

_twilio = TwilioClient(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)


def _confirm_and_send(body: str, to: str) -> str:
    print(f"\n--- PROPOSED SMS ---\nTo: {to}\n{body}\n--------------------")
    if input("Send? (yes/no): ").strip().lower() == "yes":
        msg = _twilio.messages.create(body=body, from_=TWILIO_FROM, to=to)
        return f"Sent. SID: {msg.sid}"
    return "Cancelled."


def send_sms(
    body: Annotated[str, "SMS message text"],
    to: Annotated[str, "Recipient phone in E.164 format"] = MY_PHONE,
) -> str:
    """Send an SMS after explicit user confirmation."""
    return _confirm_and_send(body, to)


sms_tool = FunctionTool.from_defaults(fn=send_sms)

# Bitly tools
bitly_client = BasicMCPClient("npx", args=["-y", "@bitly/mcp-server", "--token", BITLY_TOKEN])
bitly_spec = McpToolSpec(client=bitly_client, allowed_tools=["create_short_link", "create_qr_code"])
bitly_tools = await bitly_spec.to_tool_list_async()

print("All tools loaded.")

In [ ]:
# The full pipeline agent
pipeline_agent = ReActAgent(
    tools=canva_tools + bitly_tools + [sms_tool],
    llm=llm,
    system_prompt=(
        "You are a content pipeline agent. When asked to create and share content:\n"
        "1. Write the copy yourself\n"
        "2. Use generate-design + create-design-from-candidate to build the Canva design\n"
        "3. Use create_short_link to shorten the Canva share URL\n"
        "4. Use send_sms to text the short link to the user (after confirmation)\n"
        "Report each step's result clearly."
    ),
    verbose=True,
)

result = await pipeline_agent.run(
    f"Create an Instagram post about the benefits of running AI locally with Gemma 4. "
    f"Write the copy, design it, shorten the Canva link, and text it to {MY_PHONE}."
)
print(result)

---
## Part 4 — Descript + Canva: video thumbnail pipeline

Edit the video in Descript, then auto-generate a matching YouTube thumbnail in Canva.

In [ ]:
video_content_agent = ReActAgent(
    tools=descript_tools + canva_tools + bitly_tools + [sms_tool],
    llm=llm,
    system_prompt=(
        "You are a YouTube content agent. For each video:\n"
        "1. Use Descript to clean up the video (remove fillers, add captions, publish)\n"
        "2. Write a compelling YouTube title and thumbnail concept\n"
        "3. Use Canva to generate a youtube_thumbnail design\n"
        "4. Shorten both URLs with Bitly\n"
        "5. Text both short links to the user after confirmation\n"
    ),
    verbose=True,
)

result = await video_content_agent.run(
    f"Polish project {PROJECT_ID} in Descript (remove fillers, add captions, publish 1080p), "
    "write a YouTube title about local AI with Gemma 4, "
    f"design a bold youtube_thumbnail to match, and text both links to {MY_PHONE}."
)
print(result)

## What's next

- **Step 17 — Shopify** (if you run a store): agent writes product descriptions → creates Canva product images → publishes to your store
- **Step 25 — Vercel**: deploy a web UI so you can trigger any of these pipelines from your phone without opening a terminal
- **Step 31 — Instrumentation**: add LlamaIndex tracing to see every tool call, token count, and latency across the full pipeline